<a href="https://colab.research.google.com/github/ThayseBachmeyer/ProcessamentoSinaisI/blob/main/Aula%2005/C%C3%B3digos/prat5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Código utilizado na questão 1

In [ ]:
#n° 1

import numpy as np
from scipy.io import wavfile
from scipy.fft import fft, ifft, dct, idct
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

handel = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
nome_handel = "handel.wav"

!wget -q -O {nome_handel} {handel}

fs, x = wavfile.read(nome_handel)
if len(x.shape) > 1:
    x = x[:, 0]

x = x / np.max(np.abs(x))
N = len(x)

X_dft = fft(x)
X_dct = dct(x, norm='ortho')

energia_dft = np.abs(X_dft)**2
energia_dct = X_dct**2

coefs_ord_dft = np.sort(energia_dft)[::-1]
coefs_ord_dct = np.sort(energia_dct)[::-1]

energia_acum_dft = np.cumsum(coefs_ord_dft) / np.sum(energia_dft)
energia_acum_dct = np.cumsum(coefs_ord_dct) / np.sum(energia_dct)

fatores_r = [0.995, 0.99, 0.90, 0.75, 0.50]
mse_dft, mse_dct = [], []
tx_dft, tx_dct = [], []

for r in fatores_r:
    limiar_dft = coefs_ord_dft[np.argmax(energia_acum_dft >= r)]
    X_dft_comp = np.where(energia_dft >= limiar_dft, X_dft, 0)
    x_rec_dft = np.real(ifft(X_dft_comp))

    mse_dft.append(mean_squared_error(x, x_rec_dft))
    tx_dft.append((1 - np.sum(X_dft_comp != 0)/N) * 100)

    limiar_dct = coefs_ord_dct[np.argmax(energia_acum_dct >= r)]
    X_dct_comp = np.where(energia_dct >= limiar_dct, X_dct, 0)
    x_rec_dct = idct(X_dct_comp, norm='ortho')

    mse_dct.append(mean_squared_error(x, x_rec_dct))
    tx_dct.append((1 - np.sum(X_dct_comp != 0)/N) * 100)

fatores_pct = [r * 100 for r in fatores_r]
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(fatores_pct, mse_dft, 'ro-', linewidth=2, label='DFT')
plt.plot(fatores_pct, mse_dct, 'bo--', linewidth=2, label='DCT')
plt.xlabel('Energia Mantida (%)')
plt.ylabel('MSE (Erro Quadrático Médio)')
plt.title('Qualidade do Áudio: MSE vs Energia')
plt.gca().invert_xaxis()
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.subplot(1, 2, 2)
plt.plot(fatores_pct, tx_dft, 'ro-', linewidth=2, label='DFT')
plt.plot(fatores_pct, tx_dct, 'bo-', linewidth=2, label='DCT')
plt.xlabel('Energia Mantida (%)')
plt.ylabel('Coeficientes Descartados (%)')
plt.title('Compressão: Descarte vs Energia')
plt.gca().invert_xaxis()
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## Código utilizado na questão 2

In [ ]:
#n° 2

import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import dctn

url_img = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/sosias.jpg"
nome_img = "sosias.jpg"

!wget -q -O {nome_img} {url_img}

img = cv2.imread(nome_img, cv2.IMREAD_GRAYSCALE)
img_dct = dctn(img, norm='ortho')

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title('Imagem Original')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(np.log1p(np.abs(img_dct)), cmap='turbo')
plt.title('Energia da DCT 2-D (Escala Log)')
plt.colorbar(label='Log(Magnitude)')
plt.axis('off')

plt.tight_layout()
plt.show()

energia_total = np.sum(img_dct**2)
energia_canto = np.sum(img_dct[:50, :50]**2)
print(f"Energia no bloco 50x50 superior esquerdo: {(energia_canto / energia_total) * 100:.2f}%")

## Código utilizado na questão 3

In [ ]:
#n° 3

import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import dctn, idctn
from sklearn.metrics import mean_squared_error
import urllib.request
import os

url_img = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/sosias.jpg"
nome_img = "sosias.jpg"
!wget -q -O {nome_img} {url_img}

img = cv2.imread(nome_img, cv2.IMREAD_GRAYSCALE).astype(float)
h, w = img.shape
total_pixels = h * w

tamanhos_bloco = [8, 64]
fatores_r = [0.95, 0.50]

fig, axs = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Compressão de Imagem baseada em Blocos DCT', fontsize=16)

for i, L in enumerate(tamanhos_bloco):
    h_pad = (h // L + (1 if h % L else 0)) * L
    w_pad = (w // L + (1 if w % L else 0)) * L
    img_pad = np.pad(img, ((0, h_pad - h), (0, w_pad - w)), mode='edge')

    for j, r in enumerate(fatores_r):
        img_rec = np.zeros_like(img_pad)
        coefs_mantidos = 0

        for linha in range(0, h_pad, L):
            for col in range(0, w_pad, L):
                bloco = img_pad[linha:linha+L, col:col+L]
                bloco_dct = dctn(bloco, norm='ortho')

                energia_bloco = bloco_dct**2
                energia_total = np.sum(energia_bloco)

                if energia_total > 0:
                    coefs_ord = np.sort(energia_bloco.flatten())[::-1]
                    energia_acum = np.cumsum(coefs_ord) / energia_total
                    idx_corte = np.argmax(energia_acum >= r)
                    limiar = coefs_ord[idx_corte]
                    bloco_dct_comp = np.where(energia_bloco >= limiar, bloco_dct, 0)
                else:
                    bloco_dct_comp = np.zeros_like(bloco_dct)

                coefs_mantidos += np.sum(bloco_dct_comp != 0)
                bloco_rec = idctn(bloco_dct_comp, norm='ortho')
                img_rec[linha:linha+L, col:col+L] = bloco_rec

        img_final = np.clip(img_rec[:h, :w], 0, 255)
        mse = mean_squared_error(img, img_final)

        print(f"Bloco {L}x{L} | Energia {r*100}% -> MSE: {mse:.2f} | Coefs Mantidos: {coefs_mantidos}/{total_pixels}")

        ax = axs[i, j]
        ax.imshow(img_final, cmap='gray')
        ax.set_title(f'Bloco: {L}x{L} | Energia: {r*100}%\nMSE: {mse:.2f}')
        ax.axis('off')